# Screening a Meaty Fix for a Pea-Protein Burger

This walkthrough shows a complete food-science screening loop in Maillard: define the problem, check trust boundaries, compare two precursor strategies, generate shareable reports, and package the final decision for handoff.

The goal is not to overclaim plant-matrix quantitative certainty. The goal is to make a better next experiment.

## 1. The R&D Question

You are working on a pea-protein burger prototype built on `pea_iso`. The base is structurally acceptable, but the flavour is still weak and lacks convincing cooked-meat character.

For this screening pass, you want to answer three practical questions:

1. Does a cheap roasted-style fix such as glucose plus glycine create meaningful upside?
2. Does a sulfur-enabled system such as ribose plus cysteine create a materially stronger meaty signal?
3. Can you generate a scientist-facing package that is ready to hand to the next wet-lab decision meeting?

## 2. What We Will Compare

We hold matrix and process conditions fixed and compare two formulation hypotheses under the same cooking envelope:

- **Hypothesis A:** glucose + glycine + residual hexanal. This is a plausible roasted/nutty tweak, but it does not provide sulfur for the highest-impact meaty odorants.
- **Hypothesis B:** ribose + cysteine + residual hexanal. This is the canonical sulfur-enabled route toward FFT, MFT, and related meaty compounds.

This notebook is intentionally written as a usage example. The code cells call the public API directly and keep custom logic to a minimum.

In [8]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', message='IProgress not found.*')

import pandas as pd

project_root = Path.cwd().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.conditions import ReactionConditions
from src.inverse_design import InverseDesigner
from src.reporting import generate_comparison_report, generate_campaign_report
from src.usability_reports import DomainOfValidityChecker
from src.bayesian_optimizer import FormulationOptimizer

output_root = project_root / 'results' / 'notebook_walkthrough'
output_root.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Artifacts will be written under: {output_root}')

Project root: /Users/pabloantoniomorenocasares/Developer/Maillard
Artifacts will be written under: /Users/pabloantoniomorenocasares/Developer/Maillard/results/notebook_walkthrough


## 3. Check The Trust Boundary First

Before comparing candidates, confirm what the model can and cannot claim for this exact screen. The point here is not to block work. The point is to separate directional ranking from benchmark-backed quantitative confidence.

In [9]:
checker = DomainOfValidityChecker(target_tag='meaty')

screen_precursors = ['glucose', 'glycine', 'ribose', 'cysteine', 'hexanal']
shared_screen_warnings = checker.check(
    precursor_names=screen_precursors,
    protein_type='pea_iso',
    temp_c=130.0,
    ph=6.5,
    aw=0.95,
 )

if not shared_screen_warnings:
    print('✅ This screen sits inside the strongest validated envelope.')
else:
    print('⚠️ Shared trust warnings for this screen:')
    for warning in shared_screen_warnings:
        print(f'  - {warning.message}')

print('\nInterpretation: use this notebook to rank options and decide what to test next. Do not treat `pea_iso` results as strict plant-matrix quantitative claims yet.')

⚠️ Shared trust warnings for this screen:
  - Matrix 'pea_iso' uses speculative accessibility scaling; PRIMARY benchmarks are free-precursor only.
  - Sparse benchmark analogies: glycine, hexanal lack PRIMARY quantitative validation.

Interpretation: use this notebook to rank options and decide what to test next. Do not treat `pea_iso` results as strict plant-matrix quantitative claims yet.


## 4. Hypothesis A: A Roasted Fix Without Sulfur

This candidate asks a common practical question: can a simple glucose plus glycine tweak make the burger noticeably more meaty, or does it only produce generic roasted chemistry?

In [10]:
conditions = ReactionConditions(
    pH=6.5,
    temperature_celsius=130.0,
    water_activity=0.95,
    protein_type='pea_iso',
)
designer = InverseDesigner(target_tag='meaty', minimize_tag='beany')

hyp_a = {
    'name': 'Hypothesis A (Glucose+Glycine)',
    'sugars': ['glucose'],
    'amino_acids': ['glycine'],
    'lipids': ['hexanal'],
    'molar_ratios': {'glucose': 0.6, 'glycine': 0.6, 'hexanal': 0.15},
    'ph': 6.5,
    'temp': 130.0,
    'aw': 0.95,
    'time_minutes': 20.0,
    'protein_type': 'pea_iso',
}

warnings_a = checker.check(
    precursor_names=hyp_a['sugars'] + hyp_a['amino_acids'] + hyp_a['lipids'],
    protein_type=hyp_a['protein_type'],
    temp_c=hyp_a['temp'],
    ph=hyp_a['ph'],
    aw=hyp_a['aw'],
)
res_a = designer.evaluate_single(hyp_a, conditions)

a_fft_ppb = next((float(value) for key, value in res_a.predicted_ppb.items() if 'furfurylthiol' in key.lower()), 0.0)
a_mft_ppb = next((float(value) for key, value in res_a.predicted_ppb.items() if 'methyl-3-furanthiol' in key.lower()), 0.0)
a_thiazole_ppb = next((float(value) for key, value in res_a.predicted_ppb.items() if 'pentyl-4-methylthiazole' in key.lower()), 0.0)

print(f"Candidate: {res_a.name}")
print(f"  Meaty Score: {res_a.target_score:.2f}")
print(f"  Beany Score: {res_a.off_flavour_risk:.2f}")
print(f"  Meaty targets detected: {len(res_a.detected_targets)}")
print(f"  FFT proxy (ppb): {a_fft_ppb:.3f}")
print(f"  MFT proxy (ppb): {a_mft_ppb:.3f}")
print(f"  2-pentyl-4-methylthiazole proxy (ppb): {a_thiazole_ppb:.3f}")

print('\nInterpretation: this candidate mostly fails by not generating the meaty sulfur markers we actually care about.')

Candidate: Hypothesis A (Glucose+Glycine)
  Meaty Score: 0.00
  Beany Score: 0.00
  Meaty targets detected: 0
  FFT proxy (ppb): 0.000
  MFT proxy (ppb): 0.000
  2-pentyl-4-methylthiazole proxy (ppb): 0.000

Interpretation: this candidate mostly fails by not generating the meaty sulfur markers we actually care about.


## 5. Hypothesis B: A Sulfur-Enabled Meaty Route

Now test the more chemically targeted option: ribose plus cysteine under the same matrix and process conditions. This is the route that should unlock the high-impact meaty markers.

In [11]:
hyp_b = {
    'name': 'Hypothesis B (Ribose+Cysteine)',
    'sugars': ['ribose'],
    'amino_acids': ['cysteine'],
    'lipids': ['hexanal'],
    'molar_ratios': {'ribose': 0.6, 'cysteine': 0.6, 'hexanal': 0.15},
    'ph': 6.5,
    'temp': 130.0,
    'aw': 0.95,
    'time_minutes': 20.0,
    'protein_type': 'pea_iso',
}

warnings_b = checker.check(
    precursor_names=hyp_b['sugars'] + hyp_b['amino_acids'] + hyp_b['lipids'],
    protein_type=hyp_b['protein_type'],
    temp_c=hyp_b['temp'],
    ph=hyp_b['ph'],
    aw=hyp_b['aw'],
)
res_b = designer.evaluate_single(hyp_b, conditions)

b_fft_ppb = next((float(value) for key, value in res_b.predicted_ppb.items() if 'furfurylthiol' in key.lower()), 0.0)
b_mft_ppb = next((float(value) for key, value in res_b.predicted_ppb.items() if 'methyl-3-furanthiol' in key.lower()), 0.0)
b_thiazole_ppb = next((float(value) for key, value in res_b.predicted_ppb.items() if 'pentyl-4-methylthiazole' in key.lower()), 0.0)

print(f"Candidate: {res_b.name}")
print(f"  Meaty Score: {res_b.target_score:.2f}")
print(f"  Beany Score: {res_b.off_flavour_risk:.2f}")
print(f"  Meaty targets detected: {len(res_b.detected_targets)}")
print(f"  FFT proxy (ppb): {b_fft_ppb:.3f}")
print(f"  MFT proxy (ppb): {b_mft_ppb:.3f}")
print(f"  2-pentyl-4-methylthiazole proxy (ppb): {b_thiazole_ppb:.3f}")
print(f"  Detected meaty targets: {', '.join(res_b.detected_targets)}")

print('\nInterpretation: this candidate succeeds because sulfur-enabled chemistry actually appears in the predicted target set.')

Candidate: Hypothesis B (Ribose+Cysteine)
  Meaty Score: 32.74
  Beany Score: 0.00
  Meaty targets detected: 4
  FFT proxy (ppb): 0.390
  MFT proxy (ppb): 0.162
  2-pentyl-4-methylthiazole proxy (ppb): 0.023
  Detected meaty targets: Hydrogen Sulfide, 2-pentyl-4-methylthiazole, 2-Furfurylthiol (FFT), 2-Methyl-3-furanthiol (MFT)

Interpretation: this candidate succeeds because sulfur-enabled chemistry actually appears in the predicted target set.


## 6. Optimize Around The Winning Chemistry

After identifying the right chemistry family, the next question is not whether to use ribose plus cysteine. It is how to tune the broader formulation space around that choice.

In [12]:
warnings.filterwarnings('ignore', message='IProgress not found.*')

print('Starting Bayesian sweep (10 seeded trials)...')
opt = FormulationOptimizer(
    target_tag='meaty',
    minimize_tag='beany',
    protein_type='pea_iso',
    seed=42,
)
study = opt.optimize(
    fixed_sugars=['ribose'],
    fixed_amino_acids=['cysteine'],
    fixed_lipids=['hexanal'],
    n_trials=10,
 )

print(f'Best objective value: {study.best_value:.2f}')
print(f"Best trial target score: {study.best_trial.user_attrs.get('target_score', 0.0):.2f}")
print(f"Best trial beany score: {study.best_trial.user_attrs.get('off_flavour_risk', 0.0):.2f}")
print(f"Best trial safety score: {study.best_trial.user_attrs.get('safety_score', 0.0):.2f}")

print('\nRecommended parameter set:')
for key, value in study.best_params.items():
    if isinstance(value, (int, float)):
        print(f'  {key}: {value:.3f}')
    else:
        print(f'  {key}: {value}')

Starting Bayesian sweep (10 seeded trials)...
Best objective value: 49.14
Best trial target score: 49.49
Best trial beany score: 0.00
Best trial safety score: 0.00

Recommended parameter set:
  sugar_conc: 0.056
  aa_conc_sulfur: 0.797
  aa_conc_branched: 0.291
  aa_conc_basic: 0.158
  aa_conc_other: 0.021
  ph: 3.936
  temp: 105.808
  aw: 0.863
  time_minutes: 76.123
  intervention_agent: none
  pre_processing: none


## 7. Compare The Candidates Head-To-Head

This table is the core review surface for the screen. It focuses only on metrics that actually separate the candidates in this example.

In [13]:
comparison_df = pd.DataFrame(
    [
        {
            'Candidate': res_a.name,
            'Meaty Score': round(res_a.target_score, 2),
            'Detected Meaty Targets': len(res_a.detected_targets),
            'FFT proxy (ppb)': round(a_fft_ppb, 3),
            'MFT proxy (ppb)': round(a_mft_ppb, 3),
            '2-pentyl-4-methylthiazole proxy (ppb)': round(a_thiazole_ppb, 3),
        },
        {
            'Candidate': res_b.name,
            'Meaty Score': round(res_b.target_score, 2),
            'Detected Meaty Targets': len(res_b.detected_targets),
            'FFT proxy (ppb)': round(b_fft_ppb, 3),
            'MFT proxy (ppb)': round(b_mft_ppb, 3),
            '2-pentyl-4-methylthiazole proxy (ppb)': round(b_thiazole_ppb, 3),
        },
    ]
).set_index('Candidate')

display(
    comparison_df.style
    .highlight_max(
        subset=[
            'Meaty Score',
            'Detected Meaty Targets',
            'FFT proxy (ppb)',
            'MFT proxy (ppb)',
            '2-pentyl-4-methylthiazole proxy (ppb)',
        ],
        color='lightgreen',
    )
)

print(f"Advance to the wet lab: {comparison_df['Meaty Score'].idxmax()}")
print('\nWhy this table is more useful than the previous one:')
print('- It removes columns that were constant in this example and therefore visually misleading.')
print('- It shows the actual meaty markers that differentiate the chemistry.')
print('- It matches the real decision: Hypothesis B creates a meaty sulfur package, Hypothesis A does not.')

,Meaty Score,Detected Meaty Targets,FFT proxy (ppb),MFT proxy (ppb),2-pentyl-4-methylthiazole proxy (ppb)
Candidate,,,,,
Hypothesis A (Glucose+Glycine),0.000000,0,0.000000,0.000000,0.000000
Hypothesis B (Ribose+Cysteine),32.740000,4,0.390000,0.162000,0.023000


Advance to the wet lab: Hypothesis B (Ribose+Cysteine)

Why this table is more useful than the previous one:
- It removes columns that were constant in this example and therefore visually misleading.
- It shows the actual meaty markers that differentiate the chemistry.
- It matches the real decision: Hypothesis B creates a meaty sulfur package, Hypothesis A does not.


## 8. Generate One Clear Handoff Package

To keep the output easy to understand, this walkthrough produces one main artifact directory for the screen: a comparison report plus a campaign report. We deliberately do not surface separate per-run `report.md` files here.

In [14]:
campaign_dir = output_root / 'campaign_package'
campaign_spec = {
    'campaign': {
        'name': 'Notebook Walkthrough Screen',
        'objective': 'Choose the most compelling meaty precursor package for a pea-protein burger prototype',
        'audience': 'R&D formulation review',
    },
    'shared_conditions': {
        'ph': 6.5,
        'temp_c': 130.0,
        'matrix': 'pea_iso',
        'time_minutes': 20.0,
    },
}

generate_comparison_report(
    results=[res_a, res_b],
    conditions_list=[hyp_a, hyp_b],
    warnings_list=[warnings_a, warnings_b],
    output_dir=campaign_dir,
    campaign_metadata=campaign_spec['campaign'],
)

generate_campaign_report(
    campaign_spec=campaign_spec,
    results=[res_a, res_b],
    conditions_list=[hyp_a, hyp_b],
    run_artifacts=[
        {'name': res_a.name, 'directory': 'included in comparison package'},
        {'name': res_b.name, 'directory': 'included in comparison package'},
    ],
    warnings_list=[warnings_a, warnings_b],
    output_dir=campaign_dir,
 )

print('Main walkthrough artifacts:')
print(f"  - {campaign_dir / 'comparison.md'}")
print(f"  - {campaign_dir / 'comparison.json'}")
print(f"  - {campaign_dir / 'campaign.md'}")
print(f"  - {campaign_dir / 'campaign.json'}")

Main walkthrough artifacts:
  - /Users/pabloantoniomorenocasares/Developer/Maillard/results/notebook_walkthrough/campaign_package/comparison.md
  - /Users/pabloantoniomorenocasares/Developer/Maillard/results/notebook_walkthrough/campaign_package/comparison.json
  - /Users/pabloantoniomorenocasares/Developer/Maillard/results/notebook_walkthrough/campaign_package/campaign.md
  - /Users/pabloantoniomorenocasares/Developer/Maillard/results/notebook_walkthrough/campaign_package/campaign.json


## 9. Decision

**Advance Hypothesis B to the wet lab.**

This is now a cleaner usage notebook: it calls the API directly, keeps the output package simple, and uses a comparison table that reflects the real signal in the current model.

The practical lesson is also clearer. In this screen, the model is not saying that A is bad because it has a huge beany penalty. It is saying that A fails because it never generates the sulfur-enabled meaty chemistry that B does.